In [ ]:
import numpy as np
import pandas as pd


def generate_retail_sales_data(output_path="retail_sales.csv", start_date="2024-01-01", months_count=24):
    np.random.seed(42)

    # 1. 24-month timeframe (2024-01 to 2025-12)
    months = pd.date_range(start=start_date, periods=months_count, freq="MS").strftime("%Y-%m-%d").tolist()

    # 2. Fictional Retailers with footprint and store type dynamics
    retailers = {
        "Apex Supermarkets": {"base_stores": 310, "growth": 6, "type": "supermarket"},
        "Horizon Discount": {"base_stores": 240, "growth": 10, "type": "discount"},
        "Vanguard Express": {"base_stores": 160, "growth": 4, "type": "convenience"},
        "Beacon Hypermarket": {"base_stores": 55, "growth": 1, "type": "hypermarket"}
    }

    # 3. Fictional Brands & SKUs across strict categories: CSD, Juice, Ice Tea
    products = [
        # CSD (Carbonated Soft Drinks)
        {"category": "CSD", "brand": "Fizz Up", "sku": "FIZZ_500ML", "pack_size": 0.5, "base_price": 1.99, "base_vel": 180, "season": "csd"},
        {"category": "CSD", "brand": "Fizz Up", "sku": "FIZZ_2000ML", "pack_size": 2.0, "base_price": 2.99, "base_vel": 120, "season": "csd"},
        {"category": "CSD", "brand": "Sparkle Cola", "sku": "SPK_500ML", "pack_size": 0.5, "base_price": 1.89, "base_vel": 210, "season": "csd"},
        {"category": "CSD", "brand": "Sparkle Cola", "sku": "SPK_2000ML", "pack_size": 2.0, "base_price": 2.79, "base_vel": 150, "season": "csd"},

        # Juice
        {"category": "Juice", "brand": "SunHarvest", "sku": "SH_ORANGE_1000ML", "pack_size": 1.0, "base_price": 3.49, "base_vel": 110, "season": "juice"},
        {"category": "Juice", "brand": "SunHarvest", "sku": "SH_APPLE_1000ML", "pack_size": 1.0, "base_price": 3.29, "base_vel": 95, "season": "juice"},
        {"category": "Juice", "brand": "PureNature", "sku": "PN_ORANGE_1000ML", "pack_size": 1.0, "base_price": 3.99, "base_vel": 80, "season": "juice"},
        {"category": "Juice", "brand": "PureNature", "sku": "PN_GRAPE_1000ML", "pack_size": 1.0, "base_price": 4.19, "base_vel": 65, "season": "juice"},

        # Ice Tea
        {"category": "Ice Tea", "brand": "ChillLeaf", "sku": "CL_LEMON_500ML", "pack_size": 0.5, "base_price": 2.19, "base_vel": 140, "season": "icetea"},
        {"category": "Ice Tea", "brand": "ChillLeaf", "sku": "CL_PEACH_1500ML", "pack_size": 1.5, "base_price": 3.29, "base_vel": 90, "season": "icetea"},
        {"category": "Ice Tea", "brand": "Breeze Tea", "sku": "BT_LEMON_500ML", "pack_size": 0.5, "base_price": 1.99, "base_vel": 160, "season": "icetea"},
        {"category": "Ice Tea", "brand": "Breeze Tea", "sku": "BT_GREEN_500ML", "pack_size": 0.5, "base_price": 2.09, "base_vel": 115, "season": "icetea"},
    ]

    rows = []

    for month_idx, month_str in enumerate(months):
        m_num = int(month_str.split("-")[1])

        # Category Seasonality
        csd_season = 1.25 if m_num in [6, 7, 8] else (1.20 if m_num == 12 else (0.85 if m_num in [1, 2] else 1.00))
        icetea_season = 1.55 if m_num in [6, 7, 8] else (1.25 if m_num in [5, 9] else (0.60 if m_num in [11, 12, 1, 2] else 0.90))
        juice_season = 1.20 if m_num in [11, 12, 1, 2, 3] else (0.85 if m_num in [6, 7, 8] else 1.00)

        # Macro Trend (~3.5% yearly price inflation)
        inflation_factor = 1.0 + (month_idx / 24.0) * 0.07

        for ret_name, ret_info in retailers.items():
            # Store Expansion
            monthly_growth = (ret_info["growth"] / 12.0) * month_idx
            ret_stores = int(round(ret_info["base_stores"] + monthly_growth + np.random.randint(-1, 2)))

            # Channel Multipliers
            if ret_info["type"] == "hypermarket":
                channel_vel_mult, listing_bias = 2.8, 0.94
            elif ret_info["type"] == "supermarket":
                channel_vel_mult, listing_bias = 1.2, 0.88
            elif ret_info["type"] == "discount":
                channel_vel_mult, listing_bias = 1.5, 0.78
            else: # convenience
                channel_vel_mult, listing_bias = 0.7, 0.70

            for p in products:
                # Listing Rate Adjustment by SKU Format
                if p["pack_size"] <= 0.5 and ret_info["type"] == "convenience":
                    sku_listing_rate = min(0.98, listing_bias + 0.15)
                elif p["pack_size"] >= 1.5 and ret_info["type"] == "convenience":
                    sku_listing_rate = max(0.35, listing_bias - 0.25)
                else:
                    sku_listing_rate = listing_bias + np.random.uniform(-0.05, 0.05)

                sku_stores = max(1, min(int(round(ret_stores * sku_listing_rate)), ret_stores))

                season_mult = csd_season if p["season"] == "csd" else (icetea_season if p["season"] == "icetea" else juice_season)

                # Promo Mechanics (~15% probability per SKU/month)
                is_promo = np.random.rand() < 0.15
                price_discount = np.random.uniform(0.12, 0.22) if is_promo else 0.0
                promo_lift = 1.0 + (price_discount * np.random.uniform(2.2, 3.2)) if is_promo else 1.0

                effective_price = p["base_price"] * inflation_factor * (1.0 - price_discount) * np.random.uniform(0.98, 1.02)
                base_velocity = p["base_vel"] * channel_vel_mult * season_mult * promo_lift
                actual_velocity = base_velocity * np.random.uniform(0.92, 1.08)

                units = int(round(sku_stores * actual_velocity))
                revenue = round(units * effective_price, 2)

                rows.append({
                    "month": month_str,
                    "retailer": ret_name,
                    "category": p["category"],
                    "brand": p["brand"],
                    "sku": p["sku"],
                    "units": units,
                    "revenue": revenue,
                    "sku_stores": sku_stores,
                    "retailer_stores": ret_stores,
                    "pack_size": p["pack_size"]
                })

    df = pd.DataFrame(rows)
    df = df.dropna()
    df = df[(df["units"] > 0) & (df["revenue"] > 0) & (df["sku_stores"] <= df["retailer_stores"])]
    df.to_csv(output_path, index=False, encoding="utf-8")
    return df

if __name__ == "__main__":
    generate_retail_sales_data("retail_sales.csv")

In [ ]:
!pip install arviz pymc --upgrade

In [ ]:
# Google Colab compatible PyMC retail demand model
# Input file: retail_sales.csv
# Required columns:
# month, retailer, category, brand, sku, units, revenue,
# sku_stores, retailer_stores, pack_size
#
# Notes:
# - sku_stores must be the number of stores where the SKU was listed/available.
# - pack_size must use the same physical unit for all SKUs in a category (e.g. litres).
# - This estimates historical conditional associations, not causal price effects.

import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pymc as pm
from sklearn.preprocessing import SplineTransformer

# -------------------- SETTINGS --------------------
INPUT_FILE = Path("retail_sales.csv")
OUTPUT_DIR = Path("retail_pymc_output")
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
DRAWS = 500             # use 1500 for final estimation
TUNE = 500              # use 1500 for final estimation
CHAINS = 2              # use 4 for final estimation
TARGET_ACCEPT = 0.95
N_SPLINE_KNOTS = 4      # conservative for 24 months of data
SPLINE_DEGREE = 3

PRICE_CHANGE = 0.05     # +5% price scenario
ND_CHANGE = 0.10        # +10% relative numeric-distribution scenario

COLUMNS = {
    "month": "month",
    "retailer": "retailer",
    "category": "category",
    "brand": "brand",
    "sku": "sku",
    "units": "units",
    "revenue": "revenue",
    "sku_stores": "sku_stores",
    "retailer_stores": "retailer_stores",
    "pack_size": "pack_size",
}

# -------------------- DATA HELPERS --------------------
def get_dataset(trace, group_name):
    """Works with both xarray.DataTree and legacy InferenceData."""
    if hasattr(trace, "__getitem__") and group_name in trace:
        group = trace[group_name]
        if hasattr(group, "ds"):
            return group.ds
        if hasattr(group, "dataset"):
            return group.dataset
        return group
    if hasattr(trace, group_name):
        return getattr(trace, group_name)
    raise KeyError(f"Trace has no '{group_name}' group.")


def validate_columns(raw):
    missing = [name for name in COLUMNS.values() if name not in raw.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Available: {list(raw.columns)}")


def prepare_data(raw):
    df = raw.rename(columns={source: target for target, source in COLUMNS.items()}).copy()
    df["month"] = pd.to_datetime(df["month"])

    numeric = ["units", "revenue", "sku_stores", "retailer_stores", "pack_size"]
    for col in numeric:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    invalid = (
        df[["month", "retailer", "category", "brand", "sku"]].isna().any(axis=1)
        | df[numeric].isna().any(axis=1)
        | (df["units"] <= 0)
        | (df["revenue"] <= 0)
        | (df["sku_stores"] <= 0)
        | (df["retailer_stores"] <= 0)
        | (df["pack_size"] <= 0)
        | (df["sku_stores"] > df["retailer_stores"])
    )
    if invalid.any():
        warnings.warn(f"Dropping {invalid.sum()} invalid rows.")
        df = df.loc[~invalid].copy()

    # Pack-size adjustment: physical category volume and price per physical unit.
    df["standard_volume"] = df["units"] * df["pack_size"]
    df["price_std"] = df["revenue"] / df["standard_volume"]
    df["nd"] = df["sku_stores"] / df["retailer_stores"]
    df["velocity_std"] = df["standard_volume"] / df["sku_stores"]

    # All other SKUs are the competitor basket in each retailer-category-month.
    cell = ["retailer", "category", "month"]
    df["category_revenue"] = df.groupby(cell)["revenue"].transform("sum")
    df["category_std_volume"] = df.groupby(cell)["standard_volume"].transform("sum")
    df["competitor_revenue"] = df["category_revenue"] - df["revenue"]
    df["competitor_std_volume"] = df["category_std_volume"] - df["standard_volume"]
    df["competitor_price_std"] = df["competitor_revenue"] / df["competitor_std_volume"]

    valid = (
        (df["price_std"] > 0)
        & (df["competitor_price_std"] > 0)
        & (df["nd"] > 0)
        & (df["nd"] <= 1)
        & (df["velocity_std"] > 0)
    )
    if (~valid).any():
        warnings.warn(f"Dropping {(~valid).sum()} rows without valid competitor price or distribution.")
        df = df.loc[valid].copy()

    df["log_velocity"] = np.log(df["velocity_std"])
    df["log_relative_price"] = np.log(df["price_std"] / df["competitor_price_std"])
    df["log_nd"] = np.log(df["nd"])

    scales = {}
    for col in ["log_velocity", "log_relative_price", "log_nd"]:
        mean = float(df[col].mean())
        sd = float(df[col].std(ddof=0))
        if not np.isfinite(sd) or sd == 0:
            raise ValueError(f"'{col}' has no usable variation.")
        df[f"{col}_z"] = (df[col] - mean) / sd
        scales[col] = {"mean": mean, "sd": sd}

    # Exact pack-size group; avoids arbitrary, category-inappropriate size thresholds.
    df["pack_group"] = df["pack_size"].astype(str)
    df = df.replace([np.inf, -np.inf], np.nan).dropna().copy()
    if df.empty:
        raise ValueError("No rows remain after cleaning.")
    return df, scales


def add_indices(df):
    df = df.copy()
    level_map = {}
    for col in ["retailer", "sku", "brand", "month", "pack_group"]:
        codes, levels = pd.factorize(df[col], sort=True)
        df[f"{col}_idx"] = codes.astype("int32")
        level_map[col] = levels

    df["entity"] = df["retailer"].astype(str) + "__" + df["sku"].astype(str)
    entity_codes, entity_levels = pd.factorize(df["entity"], sort=True)
    df["entity_idx"] = entity_codes.astype("int32")

    sku_info = (
        df[["sku_idx", "brand_idx", "pack_group_idx", "sku", "brand", "pack_group", "pack_size"]]
        .sort_values("sku_idx")
        .drop_duplicates("sku_idx")
    )
    if not np.array_equal(sku_info["sku_idx"].to_numpy(), np.arange(df["sku_idx"].nunique())):
        raise ValueError("SKU indexing is inconsistent.")

    meta = {
        "retailer_levels": level_map["retailer"],
        "sku_levels": level_map["sku"],
        "brand_levels": level_map["brand"],
        "month_levels": level_map["month"],
        "pack_group_levels": level_map["pack_group"],
        "entity_levels": entity_levels,
        "sku_brand_idx": sku_info["brand_idx"].to_numpy(dtype="int32"),
        "sku_pack_idx": sku_info["pack_group_idx"].to_numpy(dtype="int32"),
        "sku_info": sku_info,
    }
    return df, meta


def fit_spline(df):
    spline = SplineTransformer(
        n_knots=N_SPLINE_KNOTS,
        degree=SPLINE_DEGREE,
        include_bias=False,
        knots="quantile",
        extrapolation="linear",
    )
    basis = spline.fit_transform(df[["log_nd_z"]]).astype("float64")
    return spline, basis

# -------------------- MODEL --------------------
def build_model(df, meta, nd_basis):
    coords = {
        "observation": np.arange(len(df)),
        "retailer": meta["retailer_levels"],
        "sku": meta["sku_levels"],
        "brand": meta["brand_levels"],
        "month": meta["month_levels"].astype(str),
        "pack_group": meta["pack_group_levels"],
        "entity": meta["entity_levels"],
        "nd_basis": np.arange(nd_basis.shape[1]),
    }

    with pm.Model(coords=coords) as model:
        retailer_idx = pm.Data("retailer_idx", df["retailer_idx"].to_numpy(), dims="observation")
        sku_idx = pm.Data("sku_idx", df["sku_idx"].to_numpy(), dims="observation")
        month_idx = pm.Data("month_idx", df["month_idx"].to_numpy(), dims="observation")
        entity_idx = pm.Data("entity_idx", df["entity_idx"].to_numpy(), dims="observation")
        x_price = pm.Data("x_price", df["log_relative_price_z"].to_numpy(), dims="observation")
        x_nd = pm.Data("x_nd", nd_basis, dims=("observation", "nd_basis"))

        alpha = pm.Normal("alpha", 0, 1)

        # Entity captures persistent retailer x SKU fit. Month controls common seasonality.
        sigma_entity = pm.HalfNormal("sigma_entity", 0.5)
        entity_effect = pm.Normal("entity_offset", 0, 1, dims="entity") * sigma_entity
        sigma_month = pm.HalfNormal("sigma_month", 0.5)
        month_effect = pm.Normal("month_offset", 0, 1, dims="month") * sigma_month

        # SKU price response with brand x pack-size partial pooling.
        price_mean = pm.Normal("price_mean", -0.4, 0.5, dims=("brand", "pack_group"))
        sigma_price_sku = pm.HalfNormal("sigma_price_sku", 0.3)
        price_slope = pm.Deterministic(
            "price_slope_z",
            price_mean[meta["sku_brand_idx"], meta["sku_pack_idx"]]
            + pm.Normal("price_offset", 0, 1, dims="sku") * sigma_price_sku,
            dims="sku",
        )

        # Nonlinear numeric-distribution effect, also grouped by brand and pack size.
        nd_mean = pm.Normal("nd_mean", 0, 0.4, dims=("brand", "pack_group", "nd_basis"))
        sigma_nd_sku = pm.HalfNormal("sigma_nd_sku", 0.25)
        nd_weights = pm.Deterministic(
            "nd_weights",
            nd_mean[meta["sku_brand_idx"], meta["sku_pack_idx"], :]
            + pm.Normal("nd_offset", 0, 1, dims=("sku", "nd_basis")) * sigma_nd_sku,
            dims=("sku", "nd_basis"),
        )
        nd_effect = pm.math.sum(nd_weights[sku_idx] * x_nd, axis=1)

        mu = (
            alpha
            + entity_effect[entity_idx]
            + month_effect[month_idx]
            + price_slope[sku_idx] * x_price
            + nd_effect
        )
        sigma = pm.HalfNormal("sigma", 0.5)
        pm.Normal("velocity_z_obs", mu=mu, sigma=sigma, observed=df["log_velocity_z"].to_numpy(), dims="observation")

    return model

# -------------------- OUTPUTS --------------------
def posterior_array(trace, variable, dimensions):
    posterior = get_dataset(trace, "posterior")
    return posterior[variable].stack(sample=("chain", "draw")).transpose(*dimensions, "sample").to_numpy()


def make_scenario(trace, df, spline, scales, price_change, nd_change):
    price_slope_z = posterior_array(trace, "price_slope_z", ("sku",))
    nd_weights = posterior_array(trace, "nd_weights", ("sku", "nd_basis"))

    sku_idx = df["sku_idx"].to_numpy()
    nd_old = df["nd"].to_numpy()
    nd_new = np.clip(nd_old * (1 + nd_change), 1e-4, 1.0)

    nd_old_z = (np.log(nd_old) - scales["log_nd"]["mean"]) / scales["log_nd"]["sd"]
    nd_new_z = (np.log(nd_new) - scales["log_nd"]["mean"]) / scales["log_nd"]["sd"]
    delta_basis = spline.transform(nd_new_z.reshape(-1, 1)) - spline.transform(nd_old_z.reshape(-1, 1))

    # Both model components are on z(log velocity), then converted to log velocity.
    nd_delta_z = np.einsum("ok,oks->os", delta_basis, nd_weights[sku_idx])
    price_delta_z = price_slope_z[sku_idx] * np.log1p(price_change) / scales["log_relative_price"]["sd"]
    delta_log_velocity = (nd_delta_z + price_delta_z) * scales["log_velocity"]["sd"]

    velocity_ratio = np.exp(delta_log_velocity)
    total_volume_ratio = velocity_ratio * (nd_new / nd_old)[:, None]

    out = df[["month", "retailer", "category", "brand", "sku", "pack_size", "units", "revenue", "sku_stores", "retailer_stores", "nd", "price_std"]].copy()
    out["scenario_price_change"] = price_change
    out["scenario_nd_change"] = nd_change
    out["scenario_nd"] = nd_new
    out["volume_multiplier_p10"] = np.quantile(total_volume_ratio, 0.10, axis=1)
    out["volume_multiplier_median"] = np.median(total_volume_ratio, axis=1)
    out["volume_multiplier_p90"] = np.quantile(total_volume_ratio, 0.90, axis=1)
    out["expected_units_median"] = out["units"] * out["volume_multiplier_median"]
    out["expected_revenue_median"] = out["revenue"] * (1 + price_change) * out["volume_multiplier_median"]
    return out


def save_outputs(trace, df, meta, spline, scales):
    print(">>> Saving posterior.nc...", flush=True)
    trace.to_netcdf(OUTPUT_DIR / "posterior.nc")

    # Manual posterior summary: no ArviZ API dependency.
    beta_z = posterior_array(trace, "price_slope_z", ("sku",))
    beta_raw = beta_z * scales["log_velocity"]["sd"] / scales["log_relative_price"]["sd"]
    sku_results = meta["sku_info"].copy()
    sku_results["elasticity_p05"] = np.quantile(beta_raw, 0.05, axis=1)
    sku_results["elasticity_median"] = np.median(beta_raw, axis=1)
    sku_results["elasticity_p95"] = np.quantile(beta_raw, 0.95, axis=1)
    sku_results.to_csv(OUTPUT_DIR / "sku_price_elasticities.csv", index=False)
    print(">>> Saved sku_price_elasticities.csv", flush=True)

    # PPC diagnostics and chart without ArviZ plotting functions.
    try:
        ppc = get_dataset(trace, "posterior_predictive")
        ppc_mean = ppc["velocity_z_obs"].mean(dim=("chain", "draw")).to_numpy()
        ppc_df = pd.DataFrame({
            "observed_z": df["log_velocity_z"].to_numpy(),
            "predicted_z": ppc_mean,
        })
        ppc_df["residual_z"] = ppc_df["observed_z"] - ppc_df["predicted_z"]
        ppc_df.to_csv(OUTPUT_DIR / "ppc_diagnostics.csv", index=False)

        plt.figure(figsize=(6, 6))
        plt.scatter(ppc_df["observed_z"], ppc_df["predicted_z"], alpha=0.55)
        low = min(ppc_df["observed_z"].min(), ppc_df["predicted_z"].min())
        high = max(ppc_df["observed_z"].max(), ppc_df["predicted_z"].max())
        plt.plot([low, high], [low, high], "k--", linewidth=1)
        plt.xlabel("Observed standardized log velocity")
        plt.ylabel("Posterior mean prediction")
        plt.title("Posterior predictive check")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "posterior_predictive_check.png", dpi=150)
        plt.close()
        print(">>> Saved PPC outputs", flush=True)
    except Exception as error:
        warnings.warn(f"PPC output skipped: {error}")

    for name, price_change, nd_change in [
        ("price_plus_5pct", PRICE_CHANGE, 0.0),
        ("nd_plus_10pct", 0.0, ND_CHANGE),
        ("price_plus_5pct_nd_plus_10pct", PRICE_CHANGE, ND_CHANGE),
    ]:
        result = make_scenario(trace, df, spline, scales, price_change, nd_change)
        result.to_csv(OUTPUT_DIR / f"scenario_{name}.csv", index=False)
        print(f">>> Saved scenario_{name}.csv", flush=True)

# -------------------- RUN --------------------
def main():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE.resolve()}")

    print(">>> Reading CSV...", flush=True)
    raw = pd.read_csv(INPUT_FILE)
    print(f">>> Raw shape: {raw.shape}", flush=True)
    print(f">>> Columns: {list(raw.columns)}", flush=True)

    print(">>> Preparing data...", flush=True)
    df, scales = prepare_data(raw)
    df, meta = add_indices(df)
    print(f">>> Clean shape: {df.shape}; retailers: {df['retailer'].nunique()}; SKUs: {df['sku'].nunique()}; months: {df['month'].nunique()}", flush=True)
    df.to_parquet(OUTPUT_DIR / "model_input.parquet", index=False)

    spline, nd_basis = fit_spline(df)
    with open(OUTPUT_DIR / "nd_spline.pkl", "wb") as file:
        pickle.dump(spline, file)

    print(">>> Building model...", flush=True)
    model = build_model(df, meta, nd_basis)

    print(">>> Sampling posterior...", flush=True)
    with model:
        trace = pm.sample(
            draws=DRAWS,
            tune=TUNE,
            chains=CHAINS,
            target_accept=TARGET_ACCEPT,
            random_seed=SEED,
            return_inferencedata=True,
        )
        print(">>> Sampling posterior predictive...", flush=True)
        trace = pm.sample_posterior_predictive(
            trace,
            var_names=["velocity_z_obs"],
            random_seed=SEED,
            extend_inferencedata=True,
        )

    save_outputs(trace, df, meta, spline, scales)
    print(f">>> DONE. Results: {OUTPUT_DIR.resolve()}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# Google Colab visualisation script for retail_pymc_colab_clean.py outputs.
# Run AFTER the PyMC model script. It displays plots inline and saves PNG/CSV files.
# Covers the full hierarchy: total market -> category -> retailer -> brand -> SKU -> pack size.

from pathlib import Path

import seaborn as sns
from IPython.display import display

OUTPUT_DIR = Path("retail_pymc_output")
MODEL_INPUT_FILE = OUTPUT_DIR / "model_input.parquet"
ELASTICITY_FILE = OUTPUT_DIR / "sku_price_elasticities.csv"
PPC_FILE = OUTPUT_DIR / "ppc_diagnostics.csv"
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(exist_ok=True)

SCENARIO_FILES = {
    "Price +5%": OUTPUT_DIR / "scenario_price_plus_5pct.csv",
    "ND +10%": OUTPUT_DIR / "scenario_nd_plus_10pct.csv",
    "Price +5% and ND +10%": OUTPUT_DIR / "scenario_price_plus_5pct_nd_plus_10pct.csv",
}

sns.set_theme(style="whitegrid", context="notebook")


def save_show(filename):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()


def month_label(value):
    return pd.Timestamp(value).strftime("%Y-%m")


def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}. Run the PyMC model first.")


def normalize_columns(data):
    """Supports all output-name variants from earlier model scripts."""
    df = data.copy()

    if "comp_price_std" not in df.columns and "competitor_price_std" in df.columns:
        df["comp_price_std"] = df["competitor_price_std"]
    if "nd" not in df.columns and "numeric_distribution" in df.columns:
        df["nd"] = df["numeric_distribution"]
    if "numeric_distribution" not in df.columns and "nd" in df.columns:
        df["numeric_distribution"] = df["nd"]
    if "standard_volume" not in df.columns:
        df["standard_volume"] = df["units"] * df["pack_size"]
    if "price_std" not in df.columns:
        df["price_std"] = df["revenue"] / df["standard_volume"]
    if "velocity_std" not in df.columns:
        df["velocity_std"] = df["standard_volume"] / df["sku_stores"]
    if "relative_price" not in df.columns:
        df["relative_price"] = df["price_std"] / df["comp_price_std"]

    required = [
        "month", "retailer", "category", "brand", "sku", "units", "revenue",
        "pack_size", "sku_stores", "retailer_stores", "standard_volume",
        "price_std", "comp_price_std", "nd", "velocity_std", "relative_price",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing fields after compatibility mapping: {missing}\nAvailable: {list(df.columns)}")
    return df


def plot_share_trend(data, level, title, filename):
    grouped = data.groupby(["month", level], as_index=False)["revenue"].sum()
    totals = grouped.groupby("month", as_index=False)["revenue"].sum().rename(columns={"revenue": "market_revenue"})
    grouped = grouped.merge(totals, on="month", how="left")
    grouped["value_share"] = grouped["revenue"] / grouped["market_revenue"]

    plt.figure(figsize=(13, 6))
    sns.lineplot(data=grouped, x="month", y="value_share", hue=level, marker="o")
    plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    plt.title(title)
    plt.xlabel("Month")
    plt.ylabel("Value share of total market")
    save_show(filename)
    return grouped


def scenario_summary(path):
    scenario = pd.read_csv(path)
    expected_units = "expected_units_median" if "expected_units_median" in scenario.columns else "units_median"
    expected_revenue = "expected_revenue_median" if "expected_revenue_median" in scenario.columns else "revenue_median"
    required = ["category", "retailer", "brand", "sku", "pack_size", "units", "revenue", expected_units, expected_revenue]
    missing = [c for c in required if c not in scenario.columns]
    if missing:
        raise ValueError(f"Scenario file {path.name} missing: {missing}")

    result = scenario.groupby(["category", "retailer", "brand", "sku", "pack_size"], as_index=False).agg(
        baseline_units=("units", "sum"),
        expected_units=(expected_units, "sum"),
        baseline_revenue=("revenue", "sum"),
        expected_revenue=(expected_revenue, "sum"),
    )
    result["unit_uplift_pct"] = 100 * (result["expected_units"] / result["baseline_units"] - 1)
    result["revenue_uplift_pct"] = 100 * (result["expected_revenue"] / result["baseline_revenue"] - 1)
    result["sku_label"] = result["brand"].astype(str) + " | " + result["sku"].astype(str)
    return result


# ==================== LOAD ====================
require_file(MODEL_INPUT_FILE)
df = pd.read_parquet(MODEL_INPUT_FILE)
df["month"] = pd.to_datetime(df["month"])
df = normalize_columns(df)

print(">>> Data validated.", flush=True)
print(f">>> Rows={len(df)}, Categories={df['category'].nunique()}, Retailers={df['retailer'].nunique()}, Brands={df['brand'].nunique()}, SKUs={df['sku'].nunique()}, Months={df['month'].nunique()}", flush=True)

display(pd.DataFrame({
    "dimension": ["Rows", "Months", "Categories", "Retailers", "Brands", "SKUs", "First month", "Last month"],
    "value": [len(df), df["month"].nunique(), df["category"].nunique(), df["retailer"].nunique(), df["brand"].nunique(), df["sku"].nunique(), month_label(df["month"].min()), month_label(df["month"].max())],
}))

# ==================== 1. TOTAL MARKET ====================
market = df.groupby("month", as_index=False).agg(
    units=("units", "sum"),
    revenue=("revenue", "sum"),
    standard_volume=("standard_volume", "sum"),
)
market["price_per_standard_unit"] = market["revenue"] / market["standard_volume"]
market.to_csv(PLOT_DIR / "market_monthly_summary.csv", index=False)

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
axes[0].plot(market["month"], market["units"], marker="o")
axes[0].set_title("Total market unit sales")
axes[0].set_ylabel("Units")
axes[1].plot(market["month"], market["revenue"], marker="o", color="tab:green")
axes[1].set_title("Total market revenue")
axes[1].set_ylabel("Revenue")
axes[2].plot(market["month"], market["price_per_standard_unit"], marker="o", color="tab:red")
axes[2].set_title("Total market average price per standard unit")
axes[2].set_ylabel("Price / standard unit")
axes[2].set_xlabel("Month")
save_show("01_total_market_trends.png")

# ==================== 2. CATEGORY ====================
category_month = df.groupby(["month", "category"], as_index=False).agg(
    revenue=("revenue", "sum"), units=("units", "sum"), standard_volume=("standard_volume", "sum")
)
category_month["price_per_standard_unit"] = category_month["revenue"] / category_month["standard_volume"]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=category_month, x="month", y="revenue", hue="category", marker="o", ax=axes[0])
axes[0].set_title("Revenue by category")
axes[0].set_ylabel("Revenue")
sns.lineplot(data=category_month, x="month", y="standard_volume", hue="category", marker="o", ax=axes[1])
axes[1].set_title("Physical standard volume by category")
axes[1].set_ylabel("Standard volume")
axes[1].set_xlabel("Month")
save_show("02_category_trends.png")
category_share = plot_share_trend(df, "category", "Category value share of total market", "03_category_value_share.png")
category_share.to_csv(PLOT_DIR / "category_value_share.csv", index=False)

# ==================== 3. RETAILERS ====================
retailer_month = df.groupby(["month", "retailer"], as_index=False).agg(
    revenue=("revenue", "sum"), units=("units", "sum"), standard_volume=("standard_volume", "sum")
)
retailer_month["price_per_standard_unit"] = retailer_month["revenue"] / retailer_month["standard_volume"]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=retailer_month, x="month", y="revenue", hue="retailer", marker="o", ax=axes[0])
axes[0].set_title("Market revenue by retailer")
axes[0].set_ylabel("Revenue")
sns.lineplot(data=retailer_month, x="month", y="price_per_standard_unit", hue="retailer", marker="o", ax=axes[1])
axes[1].set_title("Average standard-unit price by retailer")
axes[1].set_ylabel("Price / standard unit")
axes[1].set_xlabel("Month")
save_show("04_retailer_trends.png")
retailer_share = plot_share_trend(df, "retailer", "Retailer value share of total market", "05_retailer_value_share.png")
retailer_share.to_csv(PLOT_DIR / "retailer_value_share.csv", index=False)

# ==================== 4. BRANDS ====================
brand_month = df.groupby(["month", "brand"], as_index=False).agg(
    revenue=("revenue", "sum"), standard_volume=("standard_volume", "sum")
)
brand_month["price_per_standard_unit"] = brand_month["revenue"] / brand_month["standard_volume"]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=brand_month, x="month", y="revenue", hue="brand", marker="o", ax=axes[0])
axes[0].set_title("Market revenue by brand")
axes[0].set_ylabel("Revenue")
sns.lineplot(data=brand_month, x="month", y="price_per_standard_unit", hue="brand", marker="o", ax=axes[1])
axes[1].set_title("Average standard-unit price by brand")
axes[1].set_ylabel("Price / standard unit")
axes[1].set_xlabel("Month")
save_show("06_brand_trends.png")
brand_share = plot_share_trend(df, "brand", "Brand value share of total market", "07_brand_value_share.png")
brand_share.to_csv(PLOT_DIR / "brand_value_share.csv", index=False)

# ==================== 5. SKU AND PACK SIZE ====================
sku_summary = df.groupby(["brand", "sku", "pack_size"], as_index=False).agg(
    units=("units", "sum"), revenue=("revenue", "sum"), standard_volume=("standard_volume", "sum"),
    avg_nd=("nd", "mean"), avg_velocity=("velocity_std", "mean"), avg_price_std=("price_std", "mean")
)
sku_summary["sku_label"] = sku_summary["brand"].astype(str) + " | " + sku_summary["sku"].astype(str)
sku_summary = sku_summary.sort_values("revenue", ascending=False)
sku_summary.to_csv(PLOT_DIR / "sku_summary.csv", index=False)

plt.figure(figsize=(13, max(6, 0.45 * len(sku_summary))))
sns.barplot(data=sku_summary, y="sku_label", x="revenue", hue="pack_size", dodge=False, orient="h", palette="viridis")
plt.title("Total revenue by SKU and pack size")
plt.xlabel("Revenue")
plt.ylabel("Brand | SKU")
save_show("08_sku_pack_revenue.png")

# ==================== 6. PRICE, DISTRIBUTION, VELOCITY ====================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=df, x="nd", y="velocity_std", hue="brand", size="pack_size", alpha=0.65, ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Numeric distribution vs standard velocity")
axes[0].set_xlabel("Numeric distribution")
axes[0].set_ylabel("Standard volume per listed store")
sns.scatterplot(data=df, x="relative_price", y="velocity_std", hue="brand", size="pack_size", alpha=0.65, ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Relative standard price vs standard velocity")
axes[1].set_xlabel("Own price / competitor-basket price")
axes[1].set_ylabel("Standard volume per listed store")
save_show("09_price_distribution_velocity.png")

# ==================== 7. LATEST MONTH HEATMAPS ====================
latest_month = pd.Timestamp(df["month"].max())
latest = df[df["month"] == latest_month]
nd_heat = latest.pivot_table(index="sku", columns="retailer", values="nd", aggfunc="mean")
vel_heat = latest.pivot_table(index="sku", columns="retailer", values="velocity_std", aggfunc="mean")

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, 0.55 * df["sku"].nunique())))
sns.heatmap(nd_heat, annot=True, fmt=".0%", cmap="YlGnBu", linewidths=0.4, ax=axes[0])
axes[0].set_title(f"Numeric distribution: {month_label(latest_month)}")
axes[0].set_xlabel("Retailer")
axes[0].set_ylabel("SKU")
sns.heatmap(vel_heat, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.4, ax=axes[1])
axes[1].set_title(f"Standard velocity: {month_label(latest_month)}")
axes[1].set_xlabel("Retailer")
axes[1].set_ylabel("SKU")
save_show("10_latest_month_heatmaps.png")

# ==================== 8. GROWTH DECOMPOSITION ====================
# Q_standard = retailer_stores * ND * velocity_standard
all_months = np.sort(pd.to_datetime(df["month"]).unique())
first_month = pd.Timestamp(all_months[0])
last_month = pd.Timestamp(all_months[-1])
start = df[df["month"] == first_month].set_index(["retailer", "sku"])
end = df[df["month"] == last_month].set_index(["retailer", "sku"])
common = start.index.intersection(end.index)

if len(common):
    decomp = pd.DataFrame(index=common).reset_index()
    decomp["network"] = 100 * np.log(end.loc[common, "retailer_stores"].to_numpy() / start.loc[common, "retailer_stores"].to_numpy())
    decomp["distribution"] = 100 * np.log(end.loc[common, "nd"].to_numpy() / start.loc[common, "nd"].to_numpy())
    decomp["velocity"] = 100 * np.log(end.loc[common, "velocity_std"].to_numpy() / start.loc[common, "velocity_std"].to_numpy())
    decomp["total_standard_volume"] = 100 * np.log(end.loc[common, "standard_volume"].to_numpy() / start.loc[common, "standard_volume"].to_numpy())
    decomp["label"] = decomp["retailer"].astype(str) + " | " + decomp["sku"].astype(str)
    decomp = decomp.sort_values("total_standard_volume")
    decomp.to_csv(PLOT_DIR / "growth_decomposition.csv", index=False)

    long_decomp = decomp.melt(
        id_vars=["label", "total_standard_volume"],
        value_vars=["network", "distribution", "velocity"],
        var_name="driver", value_name="log_growth_contribution_pct"
    )
    plt.figure(figsize=(13, max(6, 0.38 * len(decomp))))
    sns.barplot(data=long_decomp, y="label", x="log_growth_contribution_pct", hue="driver", orient="h")
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title(f"Standard-volume growth decomposition: {month_label(first_month)} to {month_label(last_month)}")
    plt.xlabel("Log-growth contribution (%)")
    plt.ylabel("Retailer | SKU")
    save_show("11_growth_decomposition.png")

# ==================== 9. POSTERIOR ELASTICITIES AND PPC ====================
if ELASTICITY_FILE.exists():
    elasticities = pd.read_csv(ELASTICITY_FILE)
    needed = ["brand", "sku", "elasticity_p05", "elasticity_median", "elasticity_p95"]
    if all(c in elasticities.columns for c in needed):
        elasticities["label"] = elasticities["brand"].astype(str) + " | " + elasticities["sku"].astype(str)
        elasticities = elasticities.sort_values("elasticity_median")
        y = np.arange(len(elasticities))
        plt.figure(figsize=(12, max(6, 0.45 * len(elasticities))))
        plt.errorbar(elasticities["elasticity_median"], y,
                     xerr=[elasticities["elasticity_median"] - elasticities["elasticity_p05"], elasticities["elasticity_p95"] - elasticities["elasticity_median"]],
                     fmt="o", color="tab:blue", ecolor="steelblue", capsize=3)
        plt.axvline(0, color="black", linestyle="--", linewidth=1)
        plt.yticks(y, elasticities["label"])
        plt.xlabel("Price elasticity of standard velocity")
        plt.title("SKU price elasticity: posterior median and 90% interval")
        save_show("12_price_elasticities.png")

if PPC_FILE.exists():
    ppc = pd.read_csv(PPC_FILE)
    if all(c in ppc.columns for c in ["observed_z", "predicted_z", "residual_z"]):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.scatterplot(data=ppc, x="observed_z", y="predicted_z", alpha=0.65, ax=axes[0])
        low = min(ppc["observed_z"].min(), ppc["predicted_z"].min())
        high = max(ppc["observed_z"].max(), ppc["predicted_z"].max())
        axes[0].plot([low, high], [low, high], "k--", linewidth=1)
        axes[0].set_title("Posterior predictive check")
        axes[0].set_xlabel("Observed standardized log velocity")
        axes[0].set_ylabel("Predicted standardized log velocity")
        sns.histplot(ppc["residual_z"], bins=30, kde=True, ax=axes[1])
        axes[1].axvline(0, color="black", linestyle="--")
        axes[1].set_title("Residual distribution")
        axes[1].set_xlabel("Observed minus predicted")
        save_show("13_posterior_predictive_check.png")

# ==================== 10. SCENARIOS ====================
scenario_list = []
for scenario_name, path in SCENARIO_FILES.items():
    if path.exists():
        scenario = scenario_summary(path)
        scenario["scenario"] = scenario_name
        scenario_list.append(scenario)
    else:
        warnings.warn(f"Scenario file not found: {path.name}")

if scenario_list:
    scenarios = pd.concat(scenario_list, ignore_index=True)
    scenarios.to_csv(PLOT_DIR / "scenario_sku_retailer_summary.csv", index=False)

    sku_scenarios = scenarios.groupby(["scenario", "brand", "sku", "pack_size", "sku_label"], as_index=False).agg(
        baseline_units=("baseline_units", "sum"), expected_units=("expected_units", "sum"),
        baseline_revenue=("baseline_revenue", "sum"), expected_revenue=("expected_revenue", "sum")
    )
    sku_scenarios["unit_uplift_pct"] = 100 * (sku_scenarios["expected_units"] / sku_scenarios["baseline_units"] - 1)
    sku_scenarios["revenue_uplift_pct"] = 100 * (sku_scenarios["expected_revenue"] / sku_scenarios["baseline_revenue"] - 1)

    plt.figure(figsize=(14, max(6, 0.42 * sku_scenarios["sku_label"].nunique())))
    sns.barplot(data=sku_scenarios, y="sku_label", x="unit_uplift_pct", hue="scenario", errorbar=None, orient="h")
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("Expected volume change by SKU and scenario")
    plt.xlabel("Expected unit-volume change (%)")
    plt.ylabel("Brand | SKU")
    save_show("14_scenario_sku_uplift.png")

    portfolio = scenarios.groupby("scenario", as_index=False).agg(
        baseline_units=("baseline_units", "sum"), expected_units=("expected_units", "sum"),
        baseline_revenue=("baseline_revenue", "sum"), expected_revenue=("expected_revenue", "sum")
    )
    portfolio["unit_uplift_pct"] = 100 * (portfolio["expected_units"] / portfolio["baseline_units"] - 1)
    portfolio["revenue_uplift_pct"] = 100 * (portfolio["expected_revenue"] / portfolio["baseline_revenue"] - 1)
    portfolio.to_csv(PLOT_DIR / "scenario_total_market_summary.csv", index=False)
    print("Total-market scenario summary")
    display(portfolio)

    retailer_scenarios = scenarios.groupby(["scenario", "retailer"], as_index=False).agg(
        baseline_units=("baseline_units", "sum"), expected_units=("expected_units", "sum"),
        baseline_revenue=("baseline_revenue", "sum"), expected_revenue=("expected_revenue", "sum")
    )
    retailer_scenarios["unit_uplift_pct"] = 100 * (retailer_scenarios["expected_units"] / retailer_scenarios["baseline_units"] - 1)
    retailer_scenarios["revenue_uplift_pct"] = 100 * (retailer_scenarios["expected_revenue"] / retailer_scenarios["baseline_revenue"] - 1)
    retailer_scenarios.to_csv(PLOT_DIR / "scenario_retailer_summary.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=retailer_scenarios, x="retailer", y="unit_uplift_pct", hue="scenario", errorbar=None, ax=axes[0])
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_title("Expected volume change by retailer")
    axes[0].set_ylabel("Expected unit-volume change (%)")
    sns.barplot(data=retailer_scenarios, x="retailer", y="revenue_uplift_pct", hue="scenario", errorbar=None, ax=axes[1])
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Expected revenue change by retailer")
    axes[1].set_ylabel("Expected revenue change (%)")
    save_show("15_scenario_retailer_uplift.png")

print(f"\n>>> DONE. All charts and tables are in: {PLOT_DIR.resolve()}")